<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-04-rag/lesson-4.4-search-grounding/notebooks/GCP_Capstone_4.4_Search_Grounding.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4.4 Vertex AI Search & Google Search Grounding
**Netsetos GenAI Engineering — GCP Capstone**

Enterprise search over your data + live web grounding. Two GCP-unique RAG approaches.


## Setup


In [ ]:
# Colab may print pip dependency-conflict warnings for ydf and
# google-ai-generativelanguage (both Colab pre-installs this lesson does NOT use) —
# they are harmless. If a later cell raises a protobuf error, do
# Runtime > Restart session and re-run (packages are already installed).
!pip install -q google-genai==2.21.0 google-cloud-discoveryengine
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE
LOCATION = 'global'

from google import genai
from google.genai import types
from google.cloud import discoveryengine_v1 as discoveryengine

client = genai.Client(enterprise=True, project=PROJECT_ID, location='global')  # Gemini 3.x generation: global


## Cell 1: Google Search Grounding — Live Web RAG


In [ ]:
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='What are the latest developments in India semiconductor policy?',
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())]
    ),
)
print(response.text[:500])


## Cell 2: Extract Grounding Metadata


In [ ]:
cand = (response.candidates or [None])[0]
gm = getattr(cand, 'grounding_metadata', None)
if not gm or not gm.web_search_queries:
    # The tool is offered, not forced: on a question the model can answer from its weights it may
    # not search at all, and then there is no metadata to read. Ask about something recent.
    print('No grounding metadata: the model answered without searching.')
else:
    # What the model searched for
    print(f'Search queries: {gm.web_search_queries}')

    # Source URIs
    if gm.grounding_chunks:
        for chunk in gm.grounding_chunks:
            print(f'  Source: {chunk.web.title}')
            print(f'  URI: {chunk.web.uri}')

    # Text-to-source mappings
    if gm.grounding_supports:
        for support in gm.grounding_supports:
            print(f'  Claim: {support.segment.text[:80]}...')
            print(f'  Backed by: {support.grounding_chunk_indices}')


## Cell 3: Search Widget (Required by ToS)


In [ ]:
from IPython.display import HTML, display

if gm.search_entry_point:
    display(HTML(gm.search_entry_point.rendered_content))
else:
    print('No search widget in this response')


## Cell 4: Create Vertex AI Search Data Store


In [ ]:
# Note: This requires the Discovery Engine API enabled
# gcloud services enable discoveryengine.googleapis.com

ds_client = discoveryengine.DataStoreServiceClient()
parent = ds_client.collection_path(PROJECT_ID, LOCATION, 'default_collection')

data_store = discoveryengine.DataStore(
    display_name='DocuMind KB',
    industry_vertical=discoveryengine.IndustryVertical.GENERIC,
    solution_types=[discoveryengine.SolutionType.SOLUTION_TYPE_SEARCH],
    content_config=discoveryengine.DataStore.ContentConfig.CONTENT_REQUIRED)

try:
    op = ds_client.create_data_store(
        request=discoveryengine.CreateDataStoreRequest(
            parent=parent, data_store_id='documind-kb-test', data_store=data_store))
    result = op.result()
    print(f'Data store: {result.name}')
except Exception as e:
    print(f'Error (may already exist): {e}')


## Cell 5: Import Documents from GCS
The same bucket 4.3's setup filled with DocuMind's corpus — the real Acts as PDFs, the tenant's own documents as markdown. Run 4.3's *Create a docs bucket* cell first if `gs://{PROJECT_ID}-rag-docs/docs/` is empty.


In [ ]:
doc_client = discoveryengine.DocumentServiceClient()
branch = (f'projects/{PROJECT_ID}/locations/{LOCATION}'
          f'/collections/default_collection/dataStores/documind-kb-test'
          f'/branches/default_branch')

try:
    import_op = doc_client.import_documents(
        request=discoveryengine.ImportDocumentsRequest(
            parent=branch,
            gcs_source=discoveryengine.GcsSource(
                input_uris=[f'gs://{PROJECT_ID}-rag-docs/docs/*.pdf'],  # 4.3's bucket, PDFs only: the unstructured import takes
                                                                     # PDF/HTML/TXT/DOCX/PPTX; the tenant's .md files would each fail
                data_schema='content'),
            reconciliation_mode=discoveryengine.ImportDocumentsRequest
                .ReconciliationMode.INCREMENTAL))
    result = import_op.result(timeout=600)
    print('Documents imported')
except Exception as e:
    print(f'Error: {e}')


## Cell 6: Search with Summary

Summaries (`summary_spec`) are served from the **Enterprise engine with the LLM add-on** (`documind-search`, created in the lesson page's **Step 3**), not from the data store; the engine is not reproduced in this notebook, so this cell searches the data store directly. Without the engine the request carries a `summary_spec` the data store cannot serve: expect either an empty `resp.summary` or an error from the `except` below. To get a real summary, create an Enterprise engine with the LLM add-on **over this notebook's own data store (`documind-kb-test`)** and point `serving_config` at `.../engines/<your-engine>/servingConfigs/default_search`.


In [ ]:
search_client = discoveryengine.SearchServiceClient()
# The data-store path serves plain search. Summaries need an Enterprise engine with the
# LLM add-on, created over THIS notebook's data store (documind-kb-test) - the lesson page's
# Step 3 engine ("documind-search") is built over documind-kb and will not see these docs.
# Then point serving_config at
# f'projects/{PROJECT_ID}/locations/global/collections/default_collection/engines/<your-engine>/servingConfigs/default_search'
serving_config = search_client.serving_config_path(
    PROJECT_ID, LOCATION, 'documind-kb-test', 'default_config')

try:
    request = discoveryengine.SearchRequest(
        serving_config=serving_config,
        query='After how many years of continuous service does gratuity become payable?',
        page_size=5,
        # Snippets only. A summary_spec needs the Enterprise engine with the LLM add-on (the lesson
        # page's Step 3); sent to a plain data store's serving config it is an error, not a summary.
        content_search_spec=discoveryengine.SearchRequest.ContentSearchSpec(
            snippet_spec=discoveryengine.SearchRequest.ContentSearchSpec
                .SnippetSpec(return_snippet=True)))
    resp = search_client.search(request)
    for r in resp.results:
        doc = r.document.derived_struct_data
        snippet = (doc.get('snippets') or [{}])[0].get('snippet', '')
        print(f'  Title: {doc.get("title")} | {snippet[:100]}')
except Exception as e:
    print(f'Error: {e}')


## Cell 7: GroundedSearch Module


In [ ]:
class GroundedSearch:
    def __init__(self, project, location="global"):  # generation-only client -> global
        self.client = genai.Client(enterprise=True, project=project,
                                   location=location)
        self.request_count = 0  # search REQUESTS, not prompts
        self.total_cost = 0.0

    def search(self, question, model="gemini-3.6-flash"):
        """Web-grounded answer with citations and cost tracking."""
        response = self.client.models.generate_content(
            model=model, contents=question,
            config=types.GenerateContentConfig(
                tools=[types.Tool(google_search=types.GoogleSearch())]))

        gm = getattr((response.candidates or [None])[0], 'grounding_metadata', None)   # None when the model did not search
        # One prompt can issue several search requests - count each one
        requests = len(gm.web_search_queries) if gm and gm.web_search_queries else 0
        self.request_count += requests
        # $14 per 1,000 search requests after the 5,000 free per month (Gemini 3.x)
        self.total_cost += requests * 0.014

        # Extract citations
        citations = []
        if gm and gm.grounding_chunks:
            for chunk in gm.grounding_chunks:
                citations.append({"title": chunk.web.title,
                                  "uri": chunk.web.uri})
        return {
            "answer": response.text,
            "citations": citations,
            "queries": gm.web_search_queries if gm else [],
            "widget": gm.search_entry_point.rendered_content if gm and gm.search_entry_point else "",
        }

    def report(self):
        print(f"Search requests: {self.request_count} | Cost: ${self.total_cost:.2f}"
              f" (first 5,000 requests per month are free)")

gs = GroundedSearch(PROJECT_ID)
r = gs.search('Latest AI developments in India 2026')
print(f'Answer: {r["answer"][:200]}...')
print(f'Citations: {len(r["citations"])}')
for c in r['citations'][:3]:
    print(f'  {c["title"]}: {c["uri"]}')
gs.report()


## Cell 8: Cost Comparison Calculator


In [ ]:
def compare_costs(prompts_per_month):
    """Monthly cost of each RAG path at a given prompt volume.

    Figures verified 2026-09-03 (Gemini API pricing page for grounding;
    Vertex AI Search list prices).
    """
    q = prompts_per_month
    USD_INR = 85
    costs = {
        'DIY RAG (Firestore)': q * 0.0003 + 0,  # ~$0.0003/query, no infra
        'RAG Engine (Serverless)': q * 0.0025,  # $2.50/1K grounded calls; per-use vector storage not modelled - verify on the pricing page
        'Vertex AI Search (Standard)': q * 0.0015 + 5,  # $1.50/1K + storage
        'Vertex AI Search (Enterprise)': q * 0.004 + 5,  # $4/1K + storage
        # Google Search grounding on Gemini 3.x: billed per search REQUEST, assume 1.5 requests per prompt;
        # first 5,000 requests per month free, then $14 per 1,000 requests
        'Google Search (3.x)': max(0, q * 1.5 - 5000) / 1000 * 14,
    }
    print(f'Monthly cost comparison at {q:,} prompts/month:')
    for name, cost in sorted(costs.items(), key=lambda x: x[1]):
        inr = cost * USD_INR
        print(f'  {name}: ${cost:.2f} (Rs {inr:.0f})')

compare_costs(10000)
print()
compare_costs(100000)


## ✅ Lesson 4.4 Complete!

**Four RAG tiers compared so far in Module 4:**
- 4.1: Document AI — ingestion layer (OCR/Layout/Form)
- 4.2: DIY RAG — full control pipeline (embed→retrieve→generate)
- 4.3: RAG Engine — managed pipeline (corpus→import→query)
- 4.4: Search + Grounding — enterprise search + live web RAG

**Modules built:** document_ingestion.py, rag_engine.py, managed_rag.py, grounded_search.py

Module 4 continues: 4.5 Context Engineering, 4.6 Graph RAG, 4.7 Evaluation.

**Next: 4.5 — Context Engineering** (hybrid retrieval, Rank API reranking and budgeted packing on the DIY path)
